# 03 Experiment Analysis

This notebook summarizes experiment structure, compares overall treatment performance, examines segment-level heterogeneity, and produces business-facing recommendation tables.

**Expected processed inputs**
- `session_level_funnel.csv`
- `customer_level_features.csv`
- `purchase_master.csv`
- `event_master.csv`

**Main exported summaries**
- default variant recommendation
- segment recommendation tables
- campaign priority plan
- operating guardrails


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE_PATH = "../data/processed"

print("Loading processed inputs for experiment analysis...")

session_df = pd.read_csv(os.path.join(BASE_PATH, "session_level_funnel.csv"))
customer_df = pd.read_csv(os.path.join(BASE_PATH, "customer_level_features.csv"))
purchase_master = pd.read_csv(os.path.join(BASE_PATH, "purchase_master.csv"))
event_master = pd.read_csv(os.path.join(BASE_PATH, "event_master.csv"))

for col in ["session_start_ts", "session_end_ts"]:
    if col in session_df.columns:
        session_df[col] = pd.to_datetime(session_df[col], errors="coerce")

for col in ["timestamp"]:
    if col in purchase_master.columns:
        purchase_master[col] = pd.to_datetime(purchase_master[col], errors="coerce")
    if col in event_master.columns:
        event_master[col] = pd.to_datetime(event_master[col], errors="coerce")

num_cols_session = [
    "any_view", "any_click", "any_add_to_cart", "any_purchase", "is_converted",
    "session_campaign_id", "campaign_id", "is_campaign_exposed_session", "is_campaign_exposed"
]
for col in num_cols_session:
    if col in session_df.columns:
        session_df[col] = pd.to_numeric(session_df[col], errors="coerce").fillna(0)

num_cols_customer = [
    "total_sessions", "converted_sessions", "total_orders", "net_revenue_total",
    "gross_revenue_total", "refunded_orders", "premium_order_share",
    "discount_sensitivity_score", "total_add_to_cart", "has_purchase_history"
]
for col in num_cols_customer:
    if col in customer_df.columns:
        customer_df[col] = pd.to_numeric(customer_df[col], errors="coerce").fillna(0)

num_cols_purchase = ["matched_tx_flag", "gross_revenue_filled", "net_revenue", "refund_flag_filled"]
for col in num_cols_purchase:
    if col in purchase_master.columns:
        purchase_master[col] = pd.to_numeric(purchase_master[col], errors="coerce").fillna(0)

if "customer_status" not in customer_df.columns and "has_purchase_history" in customer_df.columns:
    customer_df["customer_status"] = np.where(customer_df["has_purchase_history"] == 1, "Existing_or_Buyer", "No_Purchase_History")

print("Loaded successfully")
print(f"session_df      : {session_df.shape}")
print(f"customer_df     : {customer_df.shape}")
print(f"purchase_master : {purchase_master.shape}")
print(f"event_master    : {event_master.shape}")


In [ ]:
# Schema check aligned with 01 / 02 outputs
required_session_cols = [
    "session_id", "customer_id", "experiment_group",
    "any_view", "any_click", "any_add_to_cart", "any_purchase",
    "is_converted", "session_campaign_id"
]
required_customer_cols = [
    "customer_id", "loyalty_tier", "has_purchase_history",
    "favorite_category", "discount_sensitivity_score",
    "total_add_to_cart", "total_orders"
]
required_purchase_cols = [
    "customer_id", "matched_tx_flag", "gross_revenue_filled",
    "net_revenue", "refund_flag_filled"
]

missing_session = [c for c in required_session_cols if c not in session_df.columns]
missing_customer = [c for c in required_customer_cols if c not in customer_df.columns]
missing_purchase = [c for c in required_purchase_cols if c not in purchase_master.columns]

print("missing_session:", missing_session)
print("missing_customer:", missing_customer)
print("missing_purchase:", missing_purchase)

if missing_session or missing_customer or missing_purchase:
    raise ValueError("Schema mismatch detected. Check missing columns above before running the rest of 03.")


In [ ]:
# ================================================================
# PART 3-1. Experiment structure check + prepare analysis table
# ================================================================

sdf = session_df.copy()
cdf = customer_df.copy()
pm = purchase_master.copy()

if "session_id" in pm.columns and "customer_id" in pm.columns:
    session_rev = (
        pm.groupby(["customer_id", "session_id"], as_index=False)
        .agg(
            orders_session=("matched_tx_flag", "sum"),
            gross_revenue_session=("gross_revenue_filled", "sum"),
            net_revenue_session=("net_revenue", "sum"),
            refunded_orders_session=("refund_flag_filled", "sum")
        )
    )
else:
    session_rev = sdf[["customer_id", "session_id"]].drop_duplicates().copy()
    session_rev["orders_session"] = 0.0
    session_rev["gross_revenue_session"] = 0.0
    session_rev["net_revenue_session"] = 0.0
    session_rev["refunded_orders_session"] = 0.0

customer_cols = [
    "customer_id", "loyalty_tier", "has_purchase_history", "customer_status",
    "favorite_category", "discount_sensitivity_score",
    "total_add_to_cart", "total_orders"
]
cust_features = cdf[customer_cols].copy()

rename_map = {
    "loyalty_tier": "cust_loyalty_tier",
    "has_purchase_history": "cust_has_purchase_history",
    "customer_status": "cust_customer_status",
    "favorite_category": "cust_favorite_category",
    "discount_sensitivity_score": "cust_discount_sensitivity_score",
    "total_add_to_cart": "cust_total_add_to_cart",
    "total_orders": "cust_total_orders"
}
cust_features = cust_features.rename(columns=rename_map)

session_ab = (
    sdf.merge(session_rev, on=["customer_id", "session_id"], how="left")
       .merge(cust_features, on="customer_id", how="left")
)

fill_zero_cols = [
    "orders_session", "gross_revenue_session", "net_revenue_session",
    "refunded_orders_session", "cust_has_purchase_history",
    "cust_discount_sensitivity_score", "cust_total_add_to_cart", "cust_total_orders"
]
for col in fill_zero_cols:
    if col in session_ab.columns:
        session_ab[col] = pd.to_numeric(session_ab[col], errors="coerce").fillna(0)

if "cust_customer_status" in session_ab.columns:
    session_ab["customer_status"] = session_ab["cust_customer_status"].fillna("Unknown")
else:
    session_ab["customer_status"] = np.where(
        session_ab["cust_has_purchase_history"] == 1,
        "Existing_or_Buyer",
        "No_Purchase_History"
    )

session_ab["problem_segment_flag"] = np.where(
    (session_ab["cust_total_add_to_cart"] > 0) & (session_ab["cust_total_orders"] == 0),
    1,
    0
)
session_ab["problem_segment"] = np.where(
    session_ab["problem_segment_flag"] == 1,
    "Cart_No_Purchase",
    "Other"
)

session_ab["loyalty_tier"] = session_ab["cust_loyalty_tier"]
session_ab["has_purchase_history"] = session_ab["cust_has_purchase_history"]
session_ab["favorite_category"] = session_ab["cust_favorite_category"]
session_ab["discount_sensitivity_score"] = session_ab["cust_discount_sensitivity_score"]

exp_size = (
    session_ab.groupby("experiment_group", dropna=False)
    .agg(
        total_sessions=("session_id", "count"),
        purchase_sessions=("any_purchase", "sum"),
        purchase_rate=("is_converted", "mean"),
        revenue_per_session=("net_revenue_session", "mean")
    )
    .reset_index()
    .sort_values("total_sessions", ascending=False)
)

print("\n==================== Treatment Group Size Summary ====================")
display(exp_size)

def balance_table(df, row_col, col_col="experiment_group"):
    tbl = pd.crosstab(df[row_col], df[col_col], normalize="columns")
    return (tbl * 100).round(2)

print("\n==================== Balance Check: traffic_source ====================")
display(balance_table(session_ab, "first_traffic_source"))

print("\n==================== Balance Check: loyalty_tier ====================")
display(balance_table(session_ab, "loyalty_tier"))

print("\n==================== Balance Check: customer_status ====================")
display(balance_table(session_ab, "customer_status"))

print("\n==================== Balance Check: problem_segment ====================")
display(balance_table(session_ab, "problem_segment"))

plt.figure(figsize=(7, 4))
plt.bar(exp_size["experiment_group"].astype(str), exp_size["total_sessions"])
plt.title("Experiment Group Sizes")
plt.xlabel("Experiment Group")
plt.ylabel("Sessions")
plt.show()


In [ ]:
# ================================================================
# PART 3-2. Overall A/B comparison (exclude Mixed)
# ================================================================

if "session_ab" not in globals():
    raise ValueError("session_ab is missing. Please run PART 3-1 first.")

ab_all = session_ab.copy()
ab_all = ab_all[ab_all["experiment_group"].isin(["Control", "Variant_A", "Variant_B"])].copy()

num_cols = [
    "any_view", "any_click", "any_add_to_cart", "any_purchase", "is_converted",
    "orders_session", "gross_revenue_session", "net_revenue_session", "refunded_orders_session"
]
for col in num_cols:
    if col in ab_all.columns:
        ab_all[col] = pd.to_numeric(ab_all[col], errors="coerce").fillna(0)

overall_ab = (
    ab_all.groupby("experiment_group", dropna=False)
    .agg(
        total_sessions=("session_id", "count"),
        view_sessions=("any_view", "sum"),
        click_sessions=("any_click", "sum"),
        cart_sessions=("any_add_to_cart", "sum"),
        purchase_sessions=("any_purchase", "sum"),
        orders_session=("orders_session", "sum"),
        gross_revenue=("gross_revenue_session", "sum"),
        net_revenue=("net_revenue_session", "sum"),
        refunded_orders=("refunded_orders_session", "sum")
    )
    .reset_index()
)

overall_ab["click_rate"] = overall_ab["click_sessions"] / overall_ab["total_sessions"]
overall_ab["cart_rate"] = overall_ab["cart_sessions"] / overall_ab["total_sessions"]
overall_ab["purchase_rate"] = overall_ab["purchase_sessions"] / overall_ab["total_sessions"]
overall_ab["view_to_click_rate"] = np.where(
    overall_ab["view_sessions"] > 0,
    overall_ab["click_sessions"] / overall_ab["view_sessions"],
    np.nan
)
overall_ab["click_to_cart_rate"] = np.where(
    overall_ab["click_sessions"] > 0,
    overall_ab["cart_sessions"] / overall_ab["click_sessions"],
    np.nan
)
overall_ab["cart_to_purchase_rate"] = np.where(
    overall_ab["cart_sessions"] > 0,
    overall_ab["purchase_sessions"] / overall_ab["cart_sessions"],
    np.nan
)
overall_ab["revenue_per_session"] = overall_ab["net_revenue"] / overall_ab["total_sessions"]
overall_ab["AOV"] = np.where(
    overall_ab["orders_session"] > 0,
    overall_ab["net_revenue"] / overall_ab["orders_session"],
    np.nan
)
overall_ab["refund_rate"] = np.where(
    overall_ab["orders_session"] > 0,
    overall_ab["refunded_orders"] / overall_ab["orders_session"],
    np.nan
)

control_row = overall_ab[overall_ab["experiment_group"] == "Control"].copy()
if len(control_row) != 1:
    raise ValueError("There must be exactly one Control group. Please check experiment_group values.")

control_click = control_row["click_rate"].iloc[0]
control_cart = control_row["cart_rate"].iloc[0]
control_purchase = control_row["purchase_rate"].iloc[0]
control_rps = control_row["revenue_per_session"].iloc[0]
control_aov = control_row["AOV"].iloc[0]
control_refund = control_row["refund_rate"].iloc[0]

overall_ab["click_rate_lift_vs_control"] = overall_ab["click_rate"] - control_click
overall_ab["cart_rate_lift_vs_control"] = overall_ab["cart_rate"] - control_cart
overall_ab["purchase_rate_lift_vs_control"] = overall_ab["purchase_rate"] - control_purchase
overall_ab["revenue_per_session_lift_vs_control"] = overall_ab["revenue_per_session"] - control_rps
overall_ab["AOV_lift_vs_control"] = overall_ab["AOV"] - control_aov
overall_ab["refund_rate_lift_vs_control"] = overall_ab["refund_rate"] - control_refund

order = ["Control", "Variant_A", "Variant_B"]
overall_ab["experiment_group"] = pd.Categorical(overall_ab["experiment_group"], categories=order, ordered=True)
overall_ab = overall_ab.sort_values("experiment_group")

print("\n==================== Overall A/B KPI Comparison ====================")
display(overall_ab)

lift_cols = [
    "experiment_group",
    "purchase_rate", "purchase_rate_lift_vs_control",
    "revenue_per_session", "revenue_per_session_lift_vs_control",
    "AOV", "AOV_lift_vs_control",
    "refund_rate", "refund_rate_lift_vs_control"
]
lift_summary = overall_ab[lift_cols].copy()

print("\n==================== Key Lift vs Control ====================")
display(lift_summary)

winner_purchase = overall_ab.sort_values("purchase_rate", ascending=False).iloc[0]["experiment_group"]
winner_rps = overall_ab.sort_values("revenue_per_session", ascending=False).iloc[0]["experiment_group"]
winner_refund = overall_ab.sort_values("refund_rate", ascending=True).iloc[0]["experiment_group"]

print("\n==================== Simple Winner Selection ====================")
print(f"Winner by purchase rate: {winner_purchase}")
print(f"Winner by revenue per session: {winner_rps}")
print(f"Winner by refund rate (lower is better): {winner_refund}")

plt.figure(figsize=(7, 4))
plt.bar(overall_ab["experiment_group"].astype(str), overall_ab["purchase_rate"])
plt.title("Purchase Rate by Experiment Group")
plt.xlabel("Experiment Group")
plt.ylabel("Purchase Rate")
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(overall_ab["experiment_group"].astype(str), overall_ab["revenue_per_session"])
plt.title("Revenue per Session by Experiment Group")
plt.xlabel("Experiment Group")
plt.ylabel("Revenue per Session")
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(overall_ab["experiment_group"].astype(str), overall_ab["AOV"])
plt.title("AOV by Experiment Group")
plt.xlabel("Experiment Group")
plt.ylabel("AOV")
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(overall_ab["experiment_group"].astype(str), overall_ab["refund_rate"])
plt.title("Refund Rate by Experiment Group")
plt.xlabel("Experiment Group")
plt.ylabel("Refund Rate")
plt.show()


In [ ]:
# ================================================================
# PART 3-3. Segment-level A/B comparison
# ================================================================

if "session_ab" not in globals():
    raise ValueError("session_ab is missing. Please run PART 3-1 first.")

seg_df = session_ab.copy()
seg_df = seg_df[seg_df["experiment_group"].isin(["Control", "Variant_A", "Variant_B"])].copy()

num_cols = [
    "any_purchase", "is_converted", "orders_session",
    "net_revenue_session", "refunded_orders_session"
]
for col in num_cols:
    if col in seg_df.columns:
        seg_df[col] = pd.to_numeric(seg_df[col], errors="coerce").fillna(0)

tier_order = ["Bronze", "Silver", "Gold", "Platinum"]
if "loyalty_tier" in seg_df.columns:
    seg_df["loyalty_tier"] = pd.Categorical(seg_df["loyalty_tier"], categories=tier_order, ordered=True)

def ab_by_segment(df, seg_col):
    out = (
        df.groupby([seg_col, "experiment_group"], dropna=False)
        .agg(
            total_sessions=("session_id", "count"),
            purchase_sessions=("any_purchase", "sum"),
            orders_session=("orders_session", "sum"),
            net_revenue=("net_revenue_session", "sum"),
            refunded_orders=("refunded_orders_session", "sum")
        )
        .reset_index()
    )

    out["purchase_rate"] = out["purchase_sessions"] / out["total_sessions"]
    out["revenue_per_session"] = out["net_revenue"] / out["total_sessions"]
    out["AOV"] = np.where(
        out["orders_session"] > 0,
        out["net_revenue"] / out["orders_session"],
        np.nan
    )
    out["refund_rate"] = np.where(
        out["orders_session"] > 0,
        out["refunded_orders"] / out["orders_session"],
        np.nan
    )

    ctrl = (
        out[out["experiment_group"] == "Control"][[seg_col, "purchase_rate", "revenue_per_session", "AOV", "refund_rate"]]
        .rename(columns={
            "purchase_rate": "control_purchase_rate",
            "revenue_per_session": "control_revenue_per_session",
            "AOV": "control_AOV",
            "refund_rate": "control_refund_rate"
        })
    )

    out = out.merge(ctrl, on=seg_col, how="left")
    out["purchase_rate_lift_vs_control"] = out["purchase_rate"] - out["control_purchase_rate"]
    out["revenue_per_session_lift_vs_control"] = out["revenue_per_session"] - out["control_revenue_per_session"]
    out["AOV_lift_vs_control"] = out["AOV"] - out["control_AOV"]
    out["refund_rate_lift_vs_control"] = out["refund_rate"] - out["control_refund_rate"]
    return out

seg_traffic = ab_by_segment(seg_df, "first_traffic_source")
seg_status = ab_by_segment(seg_df, "customer_status")
seg_loyalty = ab_by_segment(seg_df, "loyalty_tier")
seg_problem = ab_by_segment(seg_df, "problem_segment")

print("\n==================== traffic_source A/B Comparison ====================")
display(seg_traffic.sort_values(["first_traffic_source", "experiment_group"]))

print("\n==================== customer_status A/B Comparison ====================")
display(seg_status.sort_values(["customer_status", "experiment_group"]))

print("\n==================== loyalty_tier A/B Comparison ====================")
display(seg_loyalty.sort_values(["loyalty_tier", "experiment_group"]))

print("\n==================== problem_segment A/B Comparison ====================")
display(seg_problem.sort_values(["problem_segment", "experiment_group"]))

traffic_lift = seg_traffic[seg_traffic["experiment_group"] != "Control"].copy()
status_lift = seg_status[seg_status["experiment_group"] != "Control"].copy()
loyalty_lift = seg_loyalty[seg_loyalty["experiment_group"] != "Control"].copy()
problem_lift = seg_problem[seg_problem["experiment_group"] != "Control"].copy()

print("\n==================== traffic_source Lift vs Control ====================")
display(traffic_lift)

print("\n==================== customer_status Lift vs Control ====================")
display(status_lift)

print("\n==================== loyalty_tier Lift vs Control ====================")
display(loyalty_lift)

print("\n==================== problem_segment Lift vs Control ====================")
display(problem_lift)

def plot_metric_by_segment(df, seg_col, metric_col, title):
    pivot = df.pivot(index=seg_col, columns="experiment_group", values=metric_col)
    preferred = [c for c in ["Control", "Variant_A", "Variant_B"] if c in pivot.columns]
    pivot = pivot[preferred]
    pivot.plot(kind="bar", figsize=(10, 5))
    plt.title(title)
    plt.xlabel(seg_col)
    plt.ylabel(metric_col)
    plt.xticks(rotation=45)
    plt.show()

plot_metric_by_segment(seg_traffic, "first_traffic_source", "purchase_rate", "Purchase Rate by Traffic Source x Experiment Group")
plot_metric_by_segment(seg_status, "customer_status", "purchase_rate", "Purchase Rate by Customer Status x Experiment Group")
plot_metric_by_segment(seg_loyalty, "loyalty_tier", "purchase_rate", "Purchase Rate by Loyalty Tier x Experiment Group")
plot_metric_by_segment(seg_problem, "problem_segment", "purchase_rate", "Purchase Rate by Problem Segment x Experiment Group")

plot_metric_by_segment(seg_traffic, "first_traffic_source", "revenue_per_session", "Revenue/Session by Traffic Source x Experiment Group")
plot_metric_by_segment(seg_loyalty, "loyalty_tier", "revenue_per_session", "Revenue/Session by Loyalty Tier x Experiment Group")


In [ ]:
# ================================================================
# PART 3-4. Campaign performance + expected vs actual uplift
# ================================================================

if "session_ab" not in globals():
    raise ValueError("session_ab is missing. Please run PART 3-1 first.")

camp_df = session_ab.copy()
camp_df = camp_df[camp_df["experiment_group"].isin(["Control", "Variant_A", "Variant_B"])].copy()

if "session_campaign_id" not in camp_df.columns:
    raise ValueError("session_campaign_id is missing from session_ab. Check 01 outputs.")

num_cols = [
    "session_campaign_id", "any_purchase", "orders_session",
    "net_revenue_session", "refunded_orders_session"
]
for col in num_cols:
    if col in camp_df.columns:
        camp_df[col] = pd.to_numeric(camp_df[col], errors="coerce").fillna(0)

camp_df = camp_df[camp_df["session_campaign_id"] != 0].copy()
camp_df["session_campaign_id"] = camp_df["session_campaign_id"].astype(int)

use_cols = [c for c in ["campaign_id", "channel", "objective", "target_segment", "expected_uplift"] if c in event_master.columns]
camp_meta = event_master[use_cols].drop_duplicates().copy()
camp_meta["campaign_id"] = pd.to_numeric(camp_meta["campaign_id"], errors="coerce").fillna(0).astype(int)
camp_meta = camp_meta[camp_meta["campaign_id"] != 0].drop_duplicates(subset=["campaign_id"]).copy()

campaign_perf = (
    camp_df.groupby(["session_campaign_id", "experiment_group"], dropna=False)
    .agg(
        total_sessions=("session_id", "count"),
        purchase_sessions=("any_purchase", "sum"),
        orders_session=("orders_session", "sum"),
        net_revenue=("net_revenue_session", "sum"),
        refunded_orders=("refunded_orders_session", "sum")
    )
    .reset_index()
    .rename(columns={"session_campaign_id": "campaign_id"})
)

campaign_perf["purchase_rate"] = campaign_perf["purchase_sessions"] / campaign_perf["total_sessions"]
campaign_perf["revenue_per_session"] = campaign_perf["net_revenue"] / campaign_perf["total_sessions"]
campaign_perf["AOV"] = np.where(
    campaign_perf["orders_session"] > 0,
    campaign_perf["net_revenue"] / campaign_perf["orders_session"],
    np.nan
)
campaign_perf["refund_rate"] = np.where(
    campaign_perf["orders_session"] > 0,
    campaign_perf["refunded_orders"] / campaign_perf["orders_session"],
    np.nan
)
campaign_perf = campaign_perf.merge(camp_meta, on="campaign_id", how="left")

print("\n==================== Campaign x Experiment Group Performance ====================")
display(campaign_perf.sort_values(["campaign_id", "experiment_group"]).head(30))

control_tbl = (
    campaign_perf[campaign_perf["experiment_group"] == "Control"][["campaign_id", "purchase_rate", "revenue_per_session", "AOV", "refund_rate"]]
    .rename(columns={
        "purchase_rate": "control_purchase_rate",
        "revenue_per_session": "control_revenue_per_session",
        "AOV": "control_AOV",
        "refund_rate": "control_refund_rate"
    })
)

campaign_lift = campaign_perf.merge(control_tbl, on="campaign_id", how="left")
campaign_lift["purchase_rate_lift_vs_control"] = campaign_lift["purchase_rate"] - campaign_lift["control_purchase_rate"]
campaign_lift["revenue_per_session_lift_vs_control"] = campaign_lift["revenue_per_session"] - campaign_lift["control_revenue_per_session"]
campaign_lift["AOV_lift_vs_control"] = campaign_lift["AOV"] - campaign_lift["control_AOV"]
campaign_lift["refund_rate_lift_vs_control"] = campaign_lift["refund_rate"] - campaign_lift["control_refund_rate"]

print("\n==================== Actual Uplift by Campaign ====================")
display(campaign_lift.sort_values(["campaign_id", "experiment_group"]).head(40))

variant_only = campaign_lift[campaign_lift["experiment_group"].isin(["Variant_A", "Variant_B"])].copy()
best_campaign_variant = (
    variant_only.sort_values(
        ["campaign_id", "purchase_rate_lift_vs_control", "revenue_per_session_lift_vs_control"],
        ascending=[True, False, False]
    )
    .drop_duplicates(subset=["campaign_id"])
    .sort_values("purchase_rate_lift_vs_control", ascending=False)
)

print("\n==================== Best Variant by Campaign ====================")
display(best_campaign_variant.head(30))

channel_perf = (
    camp_df.merge(camp_meta, left_on="session_campaign_id", right_on="campaign_id", how="left")
    .groupby(["channel", "experiment_group"], dropna=False)
    .agg(
        total_sessions=("session_id", "count"),
        purchase_sessions=("any_purchase", "sum"),
        orders_session=("orders_session", "sum"),
        net_revenue=("net_revenue_session", "sum"),
        refunded_orders=("refunded_orders_session", "sum")
    )
    .reset_index()
)

channel_perf["purchase_rate"] = channel_perf["purchase_sessions"] / channel_perf["total_sessions"]
channel_perf["revenue_per_session"] = channel_perf["net_revenue"] / channel_perf["total_sessions"]
channel_perf["AOV"] = np.where(
    channel_perf["orders_session"] > 0,
    channel_perf["net_revenue"] / channel_perf["orders_session"],
    np.nan
)
channel_perf["refund_rate"] = np.where(
    channel_perf["orders_session"] > 0,
    channel_perf["refunded_orders"] / channel_perf["orders_session"],
    np.nan
)

channel_control = (
    channel_perf[channel_perf["experiment_group"] == "Control"][["channel", "purchase_rate", "revenue_per_session"]]
    .rename(columns={
        "purchase_rate": "control_purchase_rate",
        "revenue_per_session": "control_revenue_per_session"
    })
)

channel_perf = channel_perf.merge(channel_control, on="channel", how="left")
channel_perf["purchase_rate_lift_vs_control"] = channel_perf["purchase_rate"] - channel_perf["control_purchase_rate"]
channel_perf["revenue_per_session_lift_vs_control"] = channel_perf["revenue_per_session"] - channel_perf["control_revenue_per_session"]

print("\n==================== Mean Uplift by Channel ====================")
display(channel_perf.sort_values(["channel", "experiment_group"]))

objective_perf = (
    camp_df.merge(camp_meta, left_on="session_campaign_id", right_on="campaign_id", how="left")
    .groupby(["objective", "experiment_group"], dropna=False)
    .agg(
        total_sessions=("session_id", "count"),
        purchase_sessions=("any_purchase", "sum"),
        orders_session=("orders_session", "sum"),
        net_revenue=("net_revenue_session", "sum"),
        refunded_orders=("refunded_orders_session", "sum")
    )
    .reset_index()
)

objective_perf["purchase_rate"] = objective_perf["purchase_sessions"] / objective_perf["total_sessions"]
objective_perf["revenue_per_session"] = objective_perf["net_revenue"] / objective_perf["total_sessions"]
objective_perf["AOV"] = np.where(
    objective_perf["orders_session"] > 0,
    objective_perf["net_revenue"] / objective_perf["orders_session"],
    np.nan
)
objective_perf["refund_rate"] = np.where(
    objective_perf["orders_session"] > 0,
    objective_perf["refunded_orders"] / objective_perf["orders_session"],
    np.nan
)

objective_control = (
    objective_perf[objective_perf["experiment_group"] == "Control"][["objective", "purchase_rate", "revenue_per_session"]]
    .rename(columns={
        "purchase_rate": "control_purchase_rate",
        "revenue_per_session": "control_revenue_per_session"
    })
)

objective_perf = objective_perf.merge(objective_control, on="objective", how="left")
objective_perf["purchase_rate_lift_vs_control"] = objective_perf["purchase_rate"] - objective_perf["control_purchase_rate"]
objective_perf["revenue_per_session_lift_vs_control"] = objective_perf["revenue_per_session"] - objective_perf["control_revenue_per_session"]

print("\n==================== Mean Uplift by Objective ====================")
display(objective_perf.sort_values(["objective", "experiment_group"]))

compare_tbl = best_campaign_variant.copy()
compare_tbl["expected_uplift"] = pd.to_numeric(compare_tbl["expected_uplift"], errors="coerce")

corr_purchase = compare_tbl[["expected_uplift", "purchase_rate_lift_vs_control"]].corr().iloc[0, 1]
corr_rps = compare_tbl[["expected_uplift", "revenue_per_session_lift_vs_control"]].corr().iloc[0, 1]

print("\n==================== Expected vs Actual Uplift Correlation ====================")
print(f"expected_uplift vs actual purchase_rate lift correlation: {corr_purchase:.4f}")
print(f"expected_uplift vs actual revenue/session lift correlation: {corr_rps:.4f}")

plt.figure(figsize=(7, 5))
plt.scatter(compare_tbl["expected_uplift"], compare_tbl["purchase_rate_lift_vs_control"])
plt.title("Expected Uplift vs Actual Purchase Rate Lift")
plt.xlabel("Expected Uplift")
plt.ylabel("Actual Purchase Rate Lift vs Control")
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(compare_tbl["expected_uplift"], compare_tbl["revenue_per_session_lift_vs_control"])
plt.title("Expected Uplift vs Actual Revenue per Session Lift")
plt.xlabel("Expected Uplift")
plt.ylabel("Actual Revenue per Session Lift vs Control")
plt.show()


In [ ]:
# ================================================================
# PART 4A. Business Recommendation / Operating Plan
# ================================================================

required_tables = [
    "overall_ab", "seg_traffic", "seg_status",
    "seg_loyalty", "seg_problem", "best_campaign_variant"
]
missing_tables = [t for t in required_tables if t not in globals()]
if missing_tables:
    raise ValueError(f"Please run the previous parts first. Missing tables: {missing_tables}")

overall_ab_ = overall_ab.copy()
seg_traffic_ = seg_traffic.copy()
seg_status_ = seg_status.copy()
seg_loyalty_ = seg_loyalty.copy()
seg_problem_ = seg_problem.copy()
best_campaign_variant_ = best_campaign_variant.copy()

REFUND_ALERT = 0.005
REFUND_STRONG_ALERT = 0.010
PR_LIFT_EXPAND = 0.015
RPS_LIFT_EXPAND = 1.0
PR_LIFT_KEEP = 0.005
RPS_LIFT_KEEP = 0.2
SAVE_DIR = "../data/processed"
os.makedirs(SAVE_DIR, exist_ok=True)

default_tbl = overall_ab_[[
    "experiment_group",
    "purchase_rate",
    "purchase_rate_lift_vs_control",
    "revenue_per_session",
    "revenue_per_session_lift_vs_control",
    "refund_rate",
    "refund_rate_lift_vs_control"
]].copy()

def overall_decision(row):
    if row["experiment_group"] == "Control":
        return "Baseline"
    if (
        row["purchase_rate_lift_vs_control"] > 0
        and row["revenue_per_session_lift_vs_control"] > 0
        and row["refund_rate_lift_vs_control"] <= REFUND_ALERT
    ):
        return "Recommended_Default"
    if (
        row["purchase_rate_lift_vs_control"] > 0
        and row["revenue_per_session_lift_vs_control"] > 0
        and row["refund_rate_lift_vs_control"] <= REFUND_STRONG_ALERT
    ):
        return "Recommended_with_Monitoring"
    return "Not_Default"

default_tbl["overall_recommendation"] = default_tbl.apply(overall_decision, axis=1)

print("\n==================== Default Variant Recommendation ====================")
display(default_tbl)

default_candidate = default_tbl[
    default_tbl["overall_recommendation"].isin(["Recommended_Default", "Recommended_with_Monitoring"])
].sort_values(
    ["purchase_rate_lift_vs_control", "revenue_per_session_lift_vs_control"],
    ascending=False
).head(1)

if len(default_candidate) == 1:
    default_variant = default_candidate["experiment_group"].iloc[0]
else:
    default_variant = "Control"

print(f"Recommended default variant: {default_variant}")

def build_segment_reco(df, seg_col, label):
    use_cols = [
        seg_col, "experiment_group",
        "purchase_rate", "purchase_rate_lift_vs_control",
        "revenue_per_session", "revenue_per_session_lift_vs_control",
        "refund_rate", "refund_rate_lift_vs_control"
    ]
    tmp = df[use_cols].copy()
    cand = tmp[tmp["experiment_group"] != "Control"].copy()

    best = (
        cand.sort_values(
            [seg_col, "purchase_rate_lift_vs_control", "revenue_per_session_lift_vs_control"],
            ascending=[True, False, False]
        )
        .drop_duplicates(subset=[seg_col])
        .copy()
    )

    def seg_action(row):
        if (
            row["purchase_rate_lift_vs_control"] >= PR_LIFT_EXPAND
            and row["revenue_per_session_lift_vs_control"] >= RPS_LIFT_EXPAND
            and (pd.isna(row["refund_rate_lift_vs_control"]) or row["refund_rate_lift_vs_control"] <= REFUND_ALERT)
        ):
            return "Expand"
        if (
            row["purchase_rate_lift_vs_control"] > 0
            and row["revenue_per_session_lift_vs_control"] >= 0
            and (pd.isna(row["refund_rate_lift_vs_control"]) or row["refund_rate_lift_vs_control"] <= REFUND_STRONG_ALERT)
        ):
            return "Use_with_Monitoring"
        if pd.notna(row["refund_rate_lift_vs_control"]) and row["refund_rate_lift_vs_control"] > REFUND_STRONG_ALERT:
            return "Caution_Refund_Risk"
        return "Review"

    best["recommended_variant"] = best["experiment_group"]
    best["recommended_action"] = best.apply(seg_action, axis=1)
    best["segment_type"] = label

    best = best[[
        "segment_type", seg_col, "recommended_variant", "recommended_action",
        "purchase_rate_lift_vs_control", "revenue_per_session_lift_vs_control",
        "refund_rate_lift_vs_control"
    ]].rename(columns={seg_col: "segment_value"})

    return best

seg_reco_traffic = build_segment_reco(seg_traffic_, "first_traffic_source", "traffic_source")
seg_reco_status = build_segment_reco(seg_status_, "customer_status", "customer_status")
seg_reco_loyalty = build_segment_reco(seg_loyalty_, "loyalty_tier", "loyalty_tier")
seg_reco_problem = build_segment_reco(seg_problem_, "problem_segment", "problem_segment")

segment_recommendation = pd.concat(
    [seg_reco_traffic, seg_reco_status, seg_reco_loyalty, seg_reco_problem],
    ignore_index=True
)

print("\n==================== Segment-level Recommendations ====================")
display(segment_recommendation.sort_values(["segment_type", "segment_value"]))

camp = best_campaign_variant_.copy()
for col in [
    "purchase_rate_lift_vs_control",
    "revenue_per_session_lift_vs_control",
    "refund_rate_lift_vs_control",
    "expected_uplift"
]:
    if col in camp.columns:
        camp[col] = pd.to_numeric(camp[col], errors="coerce")

def campaign_priority(row):
    if (
        row["purchase_rate_lift_vs_control"] >= PR_LIFT_EXPAND
        and row["revenue_per_session_lift_vs_control"] >= RPS_LIFT_EXPAND
        and (pd.isna(row["refund_rate_lift_vs_control"]) or row["refund_rate_lift_vs_control"] <= REFUND_ALERT)
    ):
        return "Expand"
    if (
        row["purchase_rate_lift_vs_control"] >= PR_LIFT_KEEP
        and row["revenue_per_session_lift_vs_control"] >= RPS_LIFT_KEEP
        and (pd.isna(row["refund_rate_lift_vs_control"]) or row["refund_rate_lift_vs_control"] <= REFUND_STRONG_ALERT)
    ):
        return "Keep_Monitor"
    if pd.notna(row["refund_rate_lift_vs_control"]) and row["refund_rate_lift_vs_control"] > REFUND_STRONG_ALERT:
        return "Caution_Refund_Risk"
    return "Redesign_or_Low_Priority"

camp["campaign_priority"] = camp.apply(campaign_priority, axis=1)

campaign_plan = camp[[
    "campaign_id", "channel", "objective", "target_segment", "expected_uplift",
    "experiment_group", "purchase_rate_lift_vs_control",
    "revenue_per_session_lift_vs_control", "refund_rate_lift_vs_control",
    "campaign_priority"
]].sort_values(
    ["campaign_priority", "purchase_rate_lift_vs_control", "revenue_per_session_lift_vs_control"],
    ascending=[True, False, False]
)

print("\n==================== Campaign Prioritization ====================")
display(campaign_plan.head(30))

print("\n[Expand Candidates]")
display(campaign_plan[campaign_plan["campaign_priority"] == "Expand"].head(15))

print("\n[Monitor / Caution Campaigns]")
display(campaign_plan[campaign_plan["campaign_priority"].isin(["Keep_Monitor", "Caution_Refund_Risk"])].head(15))

control_row = overall_ab_[overall_ab_["experiment_group"] == "Control"].iloc[0]

guardrail_tbl = pd.DataFrame({
    "metric": [
        "purchase_rate",
        "revenue_per_session",
        "refund_rate"
    ],
    "control_baseline": [
        control_row["purchase_rate"],
        control_row["revenue_per_session"],
        control_row["refund_rate"]
    ],
    "recommended_rule": [
        "Warn if performance drops below Control",
        "Warn if performance drops below Control",
        f"Exceeds Control by +{REFUND_ALERT:.3f}p then warn"
    ],
    "hard_stop_rule": [
        "Reconsider if below Control for two consecutive weeks",
        "Reconsider if below Control for two consecutive weeks",
        f"Exceeds Control by +{REFUND_STRONG_ALERT:.3f}p then stop or redesign"
    ]
})

print("\n==================== Operating Guardrails ====================")
display(guardrail_tbl)

top_expand = campaign_plan[campaign_plan["campaign_priority"] == "Expand"].head(5)
top_expand_ids = top_expand["campaign_id"].astype(str).tolist()

summary_text = f"""
[PART 4A Recommendation Summary]

1. Default variant:
- {default_variant}

2. Core principle:
- Use {default_variant} as the default based on overall performance
- However, monitor campaigns/segments with large increases in refund_rate

3. Priority segments to expand:
- See the recommendation tables for traffic/customer/loyalty/problem segments

4. Example campaigns to expand first:
- campaign_id: {", ".join(top_expand_ids) if top_expand_ids else "None"}

5. Operating guardrails:
- Monitor purchase_rate, revenue_per_session, and refund_rate together
- Stop or redesign if refund_rate rises too far above Control
""".strip()

print("\n==================== Final Recommendation Summary ====================")
print(summary_text)

default_tbl.to_csv(os.path.join(SAVE_DIR, "part4a_default_variant_recommendation.csv"), index=False)
segment_recommendation.to_csv(os.path.join(SAVE_DIR, "part4a_segment_recommendation.csv"), index=False)
campaign_plan.to_csv(os.path.join(SAVE_DIR, "part4a_campaign_priority_plan.csv"), index=False)
guardrail_tbl.to_csv(os.path.join(SAVE_DIR, "part4a_guardrails.csv"), index=False)

with open(os.path.join(SAVE_DIR, "part4a_summary.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nSaved files:")
print("- part4a_default_variant_recommendation.csv")
print("- part4a_segment_recommendation.csv")
print("- part4a_campaign_priority_plan.csv")
print("- part4a_guardrails.csv")
print("- part4a_summary.txt")
